In [214]:
from pydantic import BaseModel
from dataclasses import dataclass, field
from typing import Any, Type

# Define prompt structure

* list of rules
* invoice schema
* few examples

In [215]:
list_rules = [
    "Use the following format to answer the question.",
    "Answer in a concise manner.",
    "If the question is not answerable, say 'I don't know'."
]

In [216]:
class InvoiceSchema(BaseModel):
    invoice_number: str
    date: str
    total_amount: float
    vendor: str

In [217]:
class Example(BaseModel):
    input: Any
    output: Any

example_1 = Example(
    input="Invoice #12345 from ABC Corp dated 2023-01-01 with total amount $1000.",
    output=InvoiceSchema(
        invoice_number="12345",
        date="2023-01-01",
        total_amount=1000.0,
        vendor="ABC Corp"
    )
)

example_2 = Example(
    input="Invoice #67890 from XYZ Inc dated 2023-02-01 with total amount $2000.",
    output=InvoiceSchema(
        invoice_number="67890",
        date="2023-02-01",
        total_amount=2000.0,
        vendor="XYZ Inc"
    )
)


list_examples = [example_1, example_2]

In [218]:
@dataclass(frozen=True)
class PromptDefinition:
    name: str
    instruction: str
    schema: Type[BaseModel]

    rules: list[str] = field(default_factory=list)
    examples: list[Example] = field(default_factory=list)

    version: str = "1.0"
    description: str | None = None
    
    
prompt_invoice_extraction_v1 = PromptDefinition(
    name="Invoice Extraction v1",
    instruction="Extract the invoice details from the given text.",
    schema=InvoiceSchema,
    rules=list_rules,
    examples=list_examples,
    version="1.0",
    description="A prompt definition for extracting invoice details from text."
)

prompt_invoice_extraction_v1

PromptDefinition(name='Invoice Extraction v1', instruction='Extract the invoice details from the given text.', schema=<class '__main__.InvoiceSchema'>, rules=['Use the following format to answer the question.', 'Answer in a concise manner.', "If the question is not answerable, say 'I don't know'."], examples=[Example(input='Invoice #12345 from ABC Corp dated 2023-01-01 with total amount $1000.', output=InvoiceSchema(invoice_number='12345', date='2023-01-01', total_amount=1000.0, vendor='ABC Corp')), Example(input='Invoice #67890 from XYZ Inc dated 2023-02-01 with total amount $2000.', output=InvoiceSchema(invoice_number='67890', date='2023-02-01', total_amount=2000.0, vendor='XYZ Inc'))], version='1.0', description='A prompt definition for extracting invoice details from text.')

# Template

In [219]:
DEFAULT_TEMPLATE = """
# TASK

{{ instruction }}

# RULES

{% for rule in rules %}
{{ loop.index }}. {{ rule }}
{% endfor %}

# SCHEMA

{{ schema }}

# EXAMPLES

{% for example in examples %}
## Example {{ loop.index }}

INPUT

{{ example.input }}

OUTPUT

{{ example.output }}

{% endfor %}

# INPUT

{{ user_query }}

# RESPONSE

Return valid JSON only.
""".strip()

In [220]:
import json
from jinja2 import Environment

class PromptRenderer:

    _env = Environment(
        trim_blocks=True,
        lstrip_blocks=True,
    )

    def render(self, user_query: str, definition: PromptDefinition) -> str:
        template = self._env.from_string(DEFAULT_TEMPLATE)

        return template.render(
            user_query=user_query,
            instruction=definition.instruction,
            rules=definition.rules,
            schema=json.dumps(definition.schema.model_json_schema(), indent=2),
            examples=[
                {
                    "input": example.input,
                    "output": json.dumps(example.output.model_dump(), indent=2),
                }
                for example in definition.examples
            ],
        )

In [221]:
renderer = PromptRenderer()

prompt =renderer.render(
    definition=prompt_invoice_extraction_v1,
    user_query="Invoice #54321 from DEF Ltd dated 2023-03-01 with total amount $1500."
)

# save prompt to file
with open("prompt_invoice_extraction_v1.txt", "w") as f:
    f.write(prompt)